Instalación de librerías para RAG local

In [ ]:

%pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers
%pip install -q transformers bitsandbytes accelerate
%pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu121
%pip install -q pypdf
%pip install -q langchain-chroma
%pip install -q patool
%pip install python-dotenv

print("¡Librerías importadas correctamente!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached https://download.pytorch.org/whl/markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cac

...


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement shutil (from versions: none)

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for shutil


Note: you may need to restart the kernel to use updated packages.
¡Librerías importadas correctamente!



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Login

In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

token = os.getenv("TOKEN")

login(token)

Descargar modelo

In [ ]:
import os
from huggingface_hub import hf_hub_download
import patoolib

repo_id = "mroscardd/BOE_spain_2026"
filename = "chroma_db_boe_2026.rar"
destino = "./"

  

print(f"Descargando dataset desde Hugging Face...")
ruta_rar = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset",
    force_download=True
)

print('Dataset descargado')


os.makedirs(destino, exist_ok=True)

print("Descomprimiendo archivo RAR...")
patoolib.extract_archive(ruta_rar, outdir=destino)


if os.path.exists(ruta_rar):
    os.remove(ruta_rar)

print("¡Proceso completado! La base de datos vectorial desde el RAR está lista.")

    

Descargando dataset desde Hugging Face...


INFO patool: Extracting C:\Users\mroscardd\.cache\huggingface\hub\datasets--mroscardd--BOE_spain_2026\snapshots\a335515e8dd25baf2a0d113e779e2d2363fe6578\chroma_db_boe_2026.rar ...
INFO patool: running 'C:\\Program Files\\WinRAR\\rar.EXE' 'x' '-kb' '-or' '--' 'C:\\Users\\mroscardd\\.cache\\huggingface\\hub\\datasets--mroscardd--BOE_spain_2026\\snapshots\\a335515e8dd25baf2a0d113e779e2d2363fe6578\\chroma_db_boe_2026.rar'


Dataset descargado
Descomprimiendo archivo RAR...


INFO patool: ... C:\Users\mroscardd\.cache\huggingface\hub\datasets--mroscardd--BOE_spain_2026\snapshots\a335515e8dd25baf2a0d113e779e2d2363fe6578\chroma_db_boe_2026.rar extracted to `./'.


¡Proceso completado! La base de datos vectorial desde el RAR está lista.


Cargar modelo en memoria

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# Cuantización a 4 bits (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Cargar Tokenizer y Modelo 
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # O el modelo que estés utilizando

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("¡Tokenizer y modelo cargados en memoria!")

Recuperar vectorStore

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'batch_size': 256, 'normalize_embeddings': True}
)


persist_directory = "./chroma_db_boe_completo"  

# Cargar la base de datos existente
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)


retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 8,
        "score_threshold": 0.35  # Filtra ruido y evita falsos positivos
    }
)

print(f"¡Base de datos cargada correctamente desde '{persist_directory}'!")



c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\Qlora_1\BOE_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22821.93it/s]


¡Base de datos cargada correctamente desde './chroma_db_boe_completo'!


Probar la búsqueda RAG + Inferencia con el modelo

In [ ]:



pregunta_usuario = input("Introduce tu pregunta")
docs_relacionados = retriever.invoke(pregunta_usuario)
contexto_leyes = "\n\n".join([f"--- Fragmento ---\n{doc.page_content}" for doc in docs_relacionados])

# Promp
messages = [
    {
        "role": "system",
        "content": (
            "Eres un asistente jurídico especializado en el ordenamiento jurídico español.\n"
            "Tu ÚNICA fuente de verdad es el CONTEXTO proporcionado.\n\n"
            "REGLAS STRICTAS:\n"
            "1. Responde a la pregunta utilizando la información explícita del CONTEXTO. "
        )
    },
    {
        "role": "user",
        "content": f"CONTEXTO LEGAL RECUPERADO:\n{contexto_leyes}\n\nPREGUNTA:\n{pregunta_usuario}"
    }
]

# Formateo
prompt_formateado = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer(prompt_formateado, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,        
        temperature=0.5,       
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

# Decodificar la respuesta
input_length = inputs.input_ids.shape[1]
respuesta_limpia = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print(respuesta_limpia)

print("--- FRAGMENTOS RECUPERADOS POR CHROMADB ---")
for i, doc in enumerate(docs_relacionados):
    print(f"\n[Fragmento {i+1}] Fuente: {doc.metadata.get('source')}")
    print(doc.page_content[:300]) # Muestra los primeros 300 caracteres